In [ ]:
#Import Libraries

import fitz
import chromadb
import torch
import numpy as np 
import pandas as pd 
import textwrap

from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

# LOAD EMBEDDING MODEL 

print("="*80)
print("LOADING EMBEDDING MODEL")
print("="*80)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(" Embedding Model Loaded Successfully!")


# LOAD MISTRAL MODEL

print("\n")
print("="*80)
print("LOADING MISTRAL MODEL")
print("="*80)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

print(" Mistral Model Loaded Successfully!")

# CREATE TEXT GENERATION PIPELINE

print("\n")
print("="*80)
print("CREATING TEXT GENERATION PIPELINE")
print("="*80)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
)

print("Pipeline Ready")

#INITIALIZE CHROMADB
print("\n")
print("="*80)
print("INITIALIZING CHROMADB")
print("="*80)

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="pdf_rage_collection",
get_or_create=True)

print("Chromadb Collection Created!")

# Loading Document
print("\n")
print("="*80)
print("LOADING PDF DOCUMENT")
print("="*80)

PDF_PATH = r"C:\Users\Mohammed\Documents\RAG PIPELINE\RAG PIPELINE\attention.pdf"

document = fitz.open(PDF_PATH)

print("PDF Loaded Successfully!")
print("Total Pages:", len(document))


# EXTRACT TEXT FROM PDF
print("\n")
print("="*80)
print("EXTRACTING TEXT")
print("="*80)

full_text= ""

for page_number in range(len(document)):
    page = document[page_number]
    text = page.get_text()
    full_text += text

print("Text Extraction Completed!")
print("Total Characters:", len(full_text))

# PREVIEW DOCUMENT TEXT
print("\n")
print("="*80)
print("DOCUMENT REVIEW")
print("="*80)

print(full_text[:1500])

# TEXT CHUNKING 
print("\n")
print("="*80)
print("CREATING TEXT CHUNKS")
print("="*80)

def chunks_text(
        text,
        chunk_size=500,
        overlap=100
):

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# GENERATE CHUNKS
chunks = chunks_text(full_text)
print("Total Chunks Created:", len(chunks))

# DISPLAY SAMPLE CHUNKS 
print("\n")
print("="*80)
print("SAMPLE CHUNKS")
print("="*80)

for i in range(min(3, len(chunks))):
    print("\n")
    print("-"*80)
    print(f"CHUNK {i+1}")
    print("-"*80)
    print(chunks[i])

# GENERATE EMBEDDINGS
print("\n")
print("="*80)
print("GENERATING EMBEDDINGS")
print("="*80)

embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Embeddings Generated Successfully!")
print("Embeddings Shape:", embeddings.shape)


# STORE EMBEDDINGS INTO CHROMADB
print("\n")
print("="*80)
print("STORING EMBEDDINGS IN CHROMADB")
print("="*80)

for index, chunk in enumerate(chunks):
    collection.add(
        ids=[str(index)], #id of specific chunk
        embeddings=[embeddings[index].tolist()],
        documents=[chunk]
    )

print("ALL Chunks Stored Successfully")

# TEST VECTOR SEARCH 
print("\n")
print("="*80)
print("TESTING VECTOR SEARCH")
print("="*80)

test_query = "What is the document about and what is the weight, key, query matrics?"
query_embedding = embedding_model.encode([test_query])
results = collection.query(query_embeddings=query_embedding.tolist(), n_results=3)
retrieved_docs = results["documents"][0]

for i, doc in enumerate(retrieved_docs):
    print("\n")
    print("-"*80)
    print(f"RETRIEVED CHUNK {i+1}")
    print("-"*80)
    print(doc[:500])

# CREATE RETRIEVAL FUNCTION
def retrieve_relevent_context(query, top_k=3):
    query_embedding = embedding_model.encode([query])

    results = collection.query(
        query_embeddings=query_embedding.tolist(), n_results=top_k
    )

    retrieved_chunks = results["documents"][0]
    contexts = "\n\n".join(retrieved_chunks)

    return contexts, results

# WITHOUT RETRIEVAL (HALLUCINATION MODE)
def ask_without_retrieval(question):
    prompt = f""" Answer the following question
            Question:
            {question}
            """
    output = generator(prompt)
    answer = output[0]["generated_text"]

    return answer

# WITH RETRIEVAL (RAG MODE)
def ask_with_retrieval(question):
    context, retrieval_results = retrieve_relevent_context(question)
    prompt = f""" You are a helpful AI assistant. Use ONLY the context below to answer the question.
    CONTEXT: {context}
    QUESTION: {question}
    
    ANSWER:
    """
    output = generator(prompt)
    answer = output[0]["generated_text"]
    return context, answer, retrieval_results


# USER QUESTION INPUT
print("\n")
print("="*80)
print("DOCUMENT QUESTION ANSWERING")
print("="*80)

question = "What is the attention mechanism, and what are the query, key, and value matrices?"


# Generate answer without retrieval
print("\n")
print("="*80)
print("ANSWER WITHOUT RETRIEVAL (POSSIBLE HALLUCINATION)")
print("="*80)

hallucinated_answer = ask_without_retrieval(question)

print(
    textwrap.fill(
        hallucinated_answer,
        width=100
    )
)


# Generate answer with retrieval
print("\n")
print("="*100)
print(" RETRIEVAL-AUGMENTED RESPONSE")
print("="*100)
context, grounded_answer, retrieval_results = ask_with_retrieval(question)
print("\n")
print("="*80)
print(" RETRIEVED CONTEXT")
print("="*80)
print(textwrap.fill(context[:2000], width=100))
print("\n")
print("="*80)
print(" GROUNDED ANSWER")
print("="*80)
print(textwrap.fill(grounded_answer, width=100))

# Similarity Score Analysis
print("\n")
print("="*80)
print("SIMILARITY SCORE ANALYSIS")
print("="*80)

distances = retrieval_results["distances"][0]

for i, distance in enumerate(distances):
    similarity_score = 1 - distance
    print(f"Chunk {i+1} Similarity Score: {similarity_score:.4f}")

# Comparision Table
print("\n")
print("="*80)
print("RAG VS HALLUCINATION COMPARISION")
print("="*80)

comparision = pd.DataFrame({
    "Approach": ["Without Retrieval", "With Retrieval"],
    "Knowledge Source": ["LLM Internal Memory", "Retrieved PDF Context"],
    "Hallucination Risk": ["High", "Low"],
    "Grounded Response": ["No", "Yes"]
}
)

print(comparision)

C:\Users\Mohammed\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING EMBEDDING MODEL


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15004.11it/s]


 Embedding Model Loaded Successfully!


LOADING MISTRAL MODEL


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:36<00:00,  7.89it/s]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


 Mistral Model Loaded Successfully!


CREATING TEXT GENERATION PIPELINE
Pipeline Ready


INITIALIZING CHROMADB
Chromadb Collection Created!


LOADING PDF DOCUMENT
PDF Loaded Successfully!
Total Pages: 15


EXTRACTING TEXT
Text Extraction Completed!
Total Characters: 39498


DOCUMENT REVIEW
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that

Batches: 100%|██████████| 4/4 [00:04<00:00,  1.01s/it]


Embeddings Generated Successfully!
Embeddings Shape: (99, 384)


STORING EMBEDDINGS IN CHROMADB
ALL Chunks Stored Successfully


TESTING VECTOR SEARCH


DOCUMENT QUESTION ANSWERING
